# CFD Backtest Pipeline — end to end

One notebook that drives the whole stack:

```
data_loader ─▶ CHOCHFibBacktester ─▶ CFDCostModel + CFDAccountSimulator ─▶ PerformanceAnalytics ─▶ ResultStore ─▶ dashboard / tv_chart
   (OHLC)         (trades+details)        (costs, equity curve)               (55 KPIs, breakdowns)     (SQLite+Parquet)     (Streamlit)
```

**Run it** in the `Market` conda env (Python 3.11), from the working dir where `../data/marketdata` and `../DB` resolve — the same place your other notebooks run. Sections:

1. Configure a run · 2. Run the whole chain in one call · 3. Inspect results · 4. Browse the store · 5. The same chain **step by step** (for debugging) · 6. Batch / parameter sweep · 7. Explore in the dashboard

Storage is SQLite + Parquet (pyarrow) only — no HDF5.


## 0 · Setup


In [ ]:
# ── environment / path bootstrap ──────────────────────────────────────────────
import os, sys
from pathlib import Path

def _find_project_root(start=None):
    # walk up from cwd until we find the dir holding libs/ and strategies/
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "libs").is_dir() and (cand / "strategies").is_dir():
            return cand
    return p

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)
print("cwd         :", os.getcwd())

# autoreload picks up edits to libs/ without a kernel restart.
# (result_store makes the pyarrow extension registration idempotent, so this is safe.)
try:
    ip = get_ipython()
    ip.run_line_magic("load_ext", "autoreload")
    ip.run_line_magic("autoreload", "2")
    ip.run_line_magic("matplotlib", "inline")
except Exception:
    pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# ── framework imports ─────────────────────────────────────────────────────────
try:                                            # pipeline.py may live at root or in libs/
    from libs.pipeline import PipelineConfig, run_pipeline, run_many
except ModuleNotFoundError:
    from pipeline import PipelineConfig, run_pipeline, run_many

from libs.result_store import ResultStore

# individual stages (used in the step-by-step section further down)
from libs import data_loader
from libs import indicators as ind
from libs.cfd_cost import CFDCostModel
from libs.account import CFDAccountSimulator
from libs.performance import PerformanceAnalytics
from strategies.strategy_msb import MSBOBFibBacktester
from strategies.hull_strategy_suit import HullSuiteStrategy
from strategies.orb_strategy import OpeningRangeBreakoutStrategy
print("imports ok")

## 1 · Configure the run

`PipelineConfig` is the single source of truth. `csv_path` resolves to
`{marketdata_path}{asset_class}:{asset}/{timeframe}.csv`, and `cost_symbol` selects the
broker cost spec (aliases like `NDX → NAS100` also work).


In [ ]:
cfg = PipelineConfig(
    run_id="GC=F-Pipe_Strategy2",
    asset="GC=F", asset_class="C", timeframe="5m",
    cost_symbol="XAUUSD",                  # broker symbol for cost specs
    initial_capital=10_000,
    use_risk_sizing=False, lots=0.1,       # fixed 0.1 lot; flip to risk sizing below
    # use_risk_sizing=True, risk_mode="percent", risk_per_trade=0.01, leverage=20,
    risk_free_rate=0.04, periods_per_year=252,
    strategy_params=dict(                  # forwarded to CHOCHFibBacktester
        open_time= "13:30",
        session_end="23:00",
        sl_mode='atr',      
        tp_mode='rr',         
        volume_filter='initial', 
        bias_hull_variation = 'hma',
        bias_source = 'close'
    ),
    marketdata_path="../data/marketdata/", db_path="../DB/",
    persist=True, overwrite=True,
)
print("csv :", cfg.csv_path)      # ../data/marketdata/I:NDX/1h.csv
print("db  :", cfg.db_file)       # ../DB/all_backtests.db
print("run :", cfg.run_id)
cfg.metadata()

## 2 · Run the whole chain (one call)

`run_pipeline` loads data → runs the backtest → prices costs and simulates the account →
computes the performance report → persists everything to the store.


In [ ]:
res = run_pipeline(cfg)
print("run_id      :", res.run_id)
print("trades      :", len(res.trades_df))
print("cost summary:", res.cost_summary)

m = res.metrics
print(f"net profit  : {m['net_profit']:.2f}    return : {m['total_return_pct']:.2f}%")
print(f"sharpe      : {m['sharpe']:.2f}    sortino: {m['sortino']:.2f}    max DD : {m['max_drawdown_pct']:.2f}%")

## 3 · Inspect the results


In [ ]:
# every KPI PerformanceAnalytics computed, as a tidy frame
pd.Series(res.metrics, name="value").to_frame()

In [ ]:
# equity curve + drawdown
ec = res.equity_curve.copy()
x = pd.to_datetime(ec["time"]) if "time" in ec and ec["time"].notna().any() else ec["step"]
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(x, ec["equity"], lw=1.3); ax[0].set_title(f"{res.run_id} — equity"); ax[0].grid(alpha=.3)
ax[1].fill_between(x, ec["drawdown_pct"], 0, color="crimson", alpha=.4)
ax[1].set_ylabel("DD %"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# exit reasons
er = res.report["exit_reasons"]
display(er)
er.set_index("exit_reason")["count"].plot.bar(figsize=(7, 3), title="exit reasons")
plt.tight_layout(); plt.show()

In [ ]:
# trades (slim) and a cost breakdown
display(res.trades_df.head())
res.costed[["spread_cost", "commission_cost", "financing_cost", "total_cost", "net_pnl"]].describe().round(2)

## 4 · It's already saved — browse the store

`run_pipeline(persist=True)` wrote this run to `../DB/all_backtests.db` (SQLite catalog +
Parquet artifacts). The dashboard reads the same file.


In [ ]:
store = ResultStore(cfg.db_file)
print("assets:", store.list_assets())
store.summary_table()

## 5 · The same chain, step by step

Exactly what `run_pipeline` does internally, one stage per cell — handy for debugging or
swapping a stage. It recomputes the same `cfg` (so the save below just overwrites the run above).


In [ ]:
# 5a · data + ATR
df = data_loader.load_csv(cfg.csv_path)
df["ATR"] = ind.calculate_atr(df, cfg.atr_length)
print("bars:", df.shape)
df.tail(3)

In [ ]:
# 5b · backtest → slim trades + rich details
bt = OpeningRangeBreakoutStrategy(run_id=cfg.run_id, **cfg.strategy_params)
trades_df, details = bt.backtest(df)
print(len(trades_df), "trades | details keys:", list(details))
trades_df.head()

In [ ]:
# 5c · details: full per-trade rows feed the account sim (they carry sl_price/tp_price/fib levels)
print({k: (len(v) if hasattr(v, "__len__") else v) for k, v in details.items() if k != "metadata"})
details["trades"].head()

In [ ]:
# 5d · CFD costs (spread / commission / overnight financing)
cost_model = CFDCostModel(cfg.cost_symbol, lots=cfg.lots)
costed = cost_model.add_costs(trades_df)
costed.head()

In [ ]:
# 5e · account simulation → result_df + equity_curve (costs applied at the sized position)
sim = CFDAccountSimulator(
    cost_model=cost_model, initial_capital=cfg.initial_capital,
    use_risk_sizing=cfg.use_risk_sizing, risk_mode=cfg.risk_mode,
    risk_per_trade=cfg.risk_per_trade, leverage=cfg.leverage,
)
result_df, equity_curve = sim.simulate(details["trades"])
print("equity:", float(equity_curve["equity"].iloc[0]), "->", round(float(equity_curve["equity"].iloc[-1]), 2))
result_df.head()

In [ ]:
# 5f · analytics → full report (metrics + breakdowns + rolling series)
perf = PerformanceAnalytics(
    result_df, equity_curve,
    risk_free_rate=cfg.risk_free_rate, periods_per_year=cfg.periods_per_year,
)
report = perf.report()
print("report keys:", list(report))
pd.Series(report["metrics"], name="value").to_frame().head(20)

In [ ]:
# 5g · persist (explicit equivalent of what run_pipeline already did)
store = ResultStore(cfg.db_file)
store.save_pipeline(
    run_id=cfg.run_id, asset=cfg.asset,
    backtest=(trades_df, details), account=(result_df, equity_curve),
    performance=report, costed=costed,
    extra_metadata=cfg.metadata(), overwrite=True,
)
store.list_runs()

## 6 · Batch several assets / sweep parameters

`run_many` opens the store once and runs every config; a failing asset is logged and skipped
(`raise_on_error=True` to stop instead). Give each run a unique `run_id` so nothing overwrites.


In [ ]:
# multi-asset batch
configs = [
    PipelineConfig(
    run_id="GC=F-Hull",
    asset="GC=F", 
    asset_class="C", 
    timeframe="5m",
    cost_symbol="XAUUSD",                  # broker symbol for cost specs
    initial_capital=10_000,
    use_risk_sizing=False, lots=0.1,       # fixed 0.1 lot; flip to risk sizing below
    periods_per_year=252,
    strategy_params=dict(                  # forwarded to CHOCHFibBacktester
        open_time= "13:30",
        session_end="23:00",
        sl_mode='atr',      
        tp_mode='rr',         
        volume_filter='initial', 
        bias_hull_variation = 'hma',
        bias_source = 'close'
    )
    ),
    
]
results = run_many(configs)
ResultStore(configs[0].db_file).summary_table()

In [10]:
# parameter sweep over real strategy kwargs → many runs, then rank
from itertools import product

grid = []
run_id = 1
for length,hull_variation,side ,atr_length,management in product([15,20,55,70,80,90,100,200],["hma","thma","ehma"],["both","long"],[14,20,30],['signal','atr']):
    run_id = run_id + 1
    grid.append(PipelineConfig(
    run_id=f"USDJPY-Hull{length}",
    asset="USDJPY=X", 
    asset_class="C", 
    timeframe='1h',
    cost_symbol="USDJPY",                  # broker symbol for cost specs
    initial_capital=10_000,
    use_risk_sizing=False, lots=0.1,       # fixed 0.1 lot; flip to risk sizing below
    periods_per_year=252,
    strategy_params=dict(
        length = length,
        hull_variation = hull_variation,
        side=side,
        atr_length=atr_length

    )
    ))

results = run_many(grid)

tbl = ResultStore(grid[0].db_file).summary_table()
tbl.sort_values("sharpe", ascending=False).head(10)

,run_id,asset,saved_at,total_trades,net_profit,win_rate,profit_factor,sharpe,sortino,max_drawdown_pct,total_return_pct,final_equity,expectancy_r
3,USDJPY-Hull70,USDJPY=X,2026-06-25T12:05:10.297048+00:00,215,124754.93,0.3581,1.182,1.398,5.359,222.35,1247.55,134754.93,0.297
4,USDJPY-Hull80,USDJPY=X,2026-06-25T12:05:39.997307+00:00,186,138992.67,0.3710,1.226,0.817,6.073,189.62,1389.93,148992.67,0.327
0,USDJPY-Hull15,USDJPY=X,2026-06-25T12:02:26.230176+00:00,659,167959.14,0.3794,1.142,0.766,14.292,138.94,1679.59,177959.14,0.100
6,USDJPY-Hull100,USDJPY=X,2026-06-25T12:06:33.281882+00:00,164,89417.09,0.3537,1.147,0.717,3.380,199.70,894.17,99417.09,0.324
1,USDJPY-Hull20,USDJPY=X,2026-06-25T12:03:54.652058+00:00,565,140922.03,0.3805,1.133,0.674,14.525,125.79,1409.22,150922.03,0.106
5,USDJPY-Hull90,USDJPY=X,2026-06-25T12:06:07.483570+00:00,178,106574.17,0.3596,1.166,0.580,4.253,195.85,1065.74,116574.17,0.304
7,USDJPY-Hull200,USDJPY=X,2026-06-25T12:06:49.526877+00:00,94,74216.03,0.3830,1.195,-0.091,-0.121,150.83,742.16,84216.03,0.306
2,USDJPY-Hull55,USDJPY=X,2026-06-25T12:04:36.477834+00:00,270,154194.52,0.3741,1.206,-1.194,-1.222,159.02,1541.95,164194.52,0.280


## 7 · Explore in the dashboard

From the project root, in the `Market` env:

```bash
streamlit run dashboard.py
# or point at a specific store:
BACKTEST_DB=../DB/all_backtests.db streamlit run dashboard.py
```

**Run detail** shows the composite score + rank, equity/drawdown, exit reasons, grouped
metric panels, long-vs-short, Monte Carlo, profit-by-session / hour, and the **TradingView
price chart** with entries / exits / TP / SL overlaid — `tv_chart.load_prices` reads the
same `../data/marketdata/{asset_class}:{asset}/{timeframe}.csv` files this notebook used.
